<a href="https://colab.research.google.com/github/chiararitorto/ppb-affinity/blob/main/PPB_Affinity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mapping Physicochemical Features onto 2D Interface Planes for Protein–Protein Binding Affinity Prediction

**Corso:** Advanced Machine Learning for Physics (A.A. 2025/2026)
**Istituzione:** Sapienza Università di Roma
**Candidata:** Chiara Ritorto

Questo notebook raccoglie ed esegue, in ordine, le fasi del progetto descritte nel report (`report/report.pdf`).
Il calcolo pesante (estrazione delle interfacce, ChimeraX, Zernike, training della CNN) non viene rilanciato qui:
il notebook carica i risultati già prodotti e ne mostra tabelle e grafici, in modo che sia eseguibile ovunque
(compreso Google Colab) senza dipendenze da ChimeraX o da un cluster HPC.


## 0. Setup dell'ambiente

In [56]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/chiararitorto/ppb-affinity.git"
    REPO_DIR = Path("/content/ppb-affinity")
    if not REPO_DIR.exists():
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
else:
    # Esecuzione locale: assume che il notebook sia nella root del repository
    REPO_DIR = Path.cwd()

sys.path.append(str(REPO_DIR))
print(f"Root del repository: {REPO_DIR}")


Root del repository: /content/ppb-affinity


In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option("display.max_columns", 20)
print("Librerie caricate correttamente.")


Librerie caricate correttamente.


## 1. Dataset

Il dataset di partenza è l'*Affinity Benchmark v5.5*, con PDB ID, catene di ligando/recettore e costante di
dissociazione K$_D$ per ciascun complesso.

In [ ]:
df_affinity = pd.read_csv(REPO_DIR / "data" / "affinity_dataset.csv", sep=";")
df_affinity.columns = [c.strip() for c in df_affinity.columns]
print(f"Numero di complessi nel dataset: {len(df_affinity)}")
df_affinity.head()


## 2. Task 1 — Interface Identification and Surface Patch Extraction

Pipeline in `src/` (moduli `pdb.py`, `interface.py`, `surface.py`, `chimerax.py`, `dataset.py`, `pipeline.py`),
eseguita tramite `scripts/task1/run_interface_extraction.py`. Identifica i residui di interfaccia
(cutoff 5.0 Å) ed estrae le patch di superficie molecolare (backend ChimeraX, `full_chain_then_filter`).

Il risultato per ciascun complesso è salvato in `outputs/task1/<indice>_<PDB_ID>/`. Qui carichiamo il
`manifest.csv` riassuntivo e un complesso di esempio (1KTZ).

In [ ]:
manifest = pd.read_csv(REPO_DIR / "outputs" / "task1" / "manifest.csv")
print(f"Complessi nel manifest: {len(manifest)}")
print(manifest['status'].value_counts())
manifest.head()


### 2.1 Selezione dei dimeri

Dal dataset completo vengono selezionati i soli complessi dimerici (una catena per ligando e per recettore),
secondo lo stesso criterio usato in `scripts/task1/filter_dimers.py`.

In [ ]:
is_dimer = (df_affinity['Ligand Chains'].astype(str).str.len() == 1) & \
           (df_affinity['Receptor Chains'].astype(str).str.len() == 1)
dimer_ids = set(df_affinity.loc[is_dimer, 'PDB'].astype(str).str.strip())

manifest_dimers = manifest[manifest['pdb_id'].isin(dimer_ids)].drop_duplicates(subset='pdb_id')
print(f"Complessi dimerici trovati: {len(manifest_dimers)}")

# Nota: 1NVU compare due volte nel dataset originale (catene ligando Q e R,
# varianti puntiformi dello stesso complesso, stesso recettore S). La pipeline
# indicizza l'output per PDB ID, quindi ne conserva una sola cartella su disco;
# da qui la deduplica esplicita sopra.


### 2.2 Esempio: complesso 1KTZ

In [ ]:
example_dir = REPO_DIR / "outputs" / "task1" / "00000_1KTZ"

with open(example_dir / "metadata.json") as f:
    meta_1ktz = json.load(f)

print(json.dumps(meta_1ktz["counts"], indent=2))


In [ ]:
ligand_patch = pd.read_csv(example_dir / "ligand_surface_patch.csv")
receptor_patch = pd.read_csv(example_dir / "receptor_surface_patch.csv")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(ligand_patch["x"], ligand_patch["y"], ligand_patch["z"], s=2, alpha=0.5, label="ligando (A)")
ax.scatter(receptor_patch["x"], receptor_patch["y"], receptor_patch["z"], s=2, alpha=0.5, label="recettore (B)")
ax.set_title("Patch di interfaccia — complesso 1KTZ")
ax.legend()
plt.tight_layout()
plt.show()


### 2.3 Statistiche aggregate sui dimeri

In [ ]:
cols = ['ligand_interface_residues', 'receptor_interface_residues', 'residue_contacts',
        'ligand_patch_atoms', 'receptor_patch_atoms',
        'ligand_surface_points', 'receptor_surface_points', 'all_atoms']

manifest_dimers[cols].describe().T[['mean', 'std', 'min', '50%', 'max']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

axes[0].hist(manifest_dimers['ligand_interface_residues'], bins=15, alpha=0.7, label='ligando', color='#4C72B0')
axes[0].hist(manifest_dimers['receptor_interface_residues'], bins=15, alpha=0.7, label='recettore', color='#DD8452')
axes[0].set_xlabel('Numero di residui di interfaccia')
axes[0].set_ylabel('Numero di complessi')
axes[0].set_title("Dimensione dell'interfaccia (dimeri)")
axes[0].legend(fontsize=8)

axes[1].hist(manifest_dimers['residue_contacts'], bins=15, color='#55A868')
axes[1].set_xlabel('Numero di contatti residuo-residuo')
axes[1].set_ylabel('Numero di complessi')
axes[1].set_title('Contatti di interfaccia (dimeri)')

plt.tight_layout()
plt.show()


## 3. Task 2 — Mappatura della complementarità tramite descrittori di Zernike

Script di produzione: `scripts/task2/run_task2_parallel.py`, eseguito su CINECA Leonardo sui 120 dimeri. **118 su 120** hanno completato il calcolo con successo (2PCC e 1EWY non hanno terminato entro i limiti di coda, anche dopo il rilancio con tempo massimo aumentato).
Per ciascun complesso i punti di superficie di ligando e recettore (Task 1) vengono sottocampionati con stride 20, espansi
in descrittori locali di Zernike (ordine 20, raggio 6.0 Å, `verso = +1` per il ligando e `-1` per il recettore), e confrontati
con un matching globale nello spazio dei descrittori (`scipy.spatial.distance.cdist`), simmetrico andata + ritorno.

Vedi la Sezione "Task 2" del report per la spiegazione completa dell'algoritmo e del costo computazionale su HPC.

In [ ]:
task2_example_dir = REPO_DIR / "outputs" / "task2" / "00000_1KTZ"

zernike_1ktz = pd.read_csv(task2_example_dir / "zernike_complementarity_symmetric.csv")
print(f"Righe totali: {len(zernike_1ktz)}")
print(zernike_1ktz['source'].value_counts())
zernike_1ktz.head()

### 3.1 Distribuzione dello score di complementarità

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for source, color in [("ligand", "#4C72B0"), ("receptor", "#DD8452")]:
    subset = zernike_1ktz[zernike_1ktz["source"] == source]
    ax.hist(subset["complementarity_score"], bins=25, alpha=0.6, label=source, color=color)

ax.set_xlabel("Complementarity score (= -distanza Zernike)")
ax.set_ylabel("Numero di punti")
ax.set_title("Distribuzione dello score di complementarità — 1KTZ")
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Visualizzazione spaziale dei punti campionati, colorati per score

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

sc = ax.scatter(
    zernike_1ktz["x"], zernike_1ktz["y"], zernike_1ktz["z"],
    c=zernike_1ktz["complementarity_score"], cmap="viridis", s=15
)
ax.set_xlabel("x (Å)")
ax.set_ylabel("y (Å)")
ax.set_zlabel("z (Å)")
ax.set_title("Punti campionati e score di complementarità — 1KTZ")
fig.colorbar(sc, ax=ax, label="complementarity score", shrink=0.6)
plt.tight_layout()
plt.show()

## 4. Task 3 — Costruzione dei piani di complementarità 2D

Script: `scripts/task3/run_task3_projection.py` (proiezione PCA), `scripts/task3/run_task3_lennard_jones.py` (filtro energetico
e correlazione), `scripts/task3/generate_interface_images.py` (costruzione dei tensori 2D).

**Nota sul dataset**: dei 120 dimeri, solo i 118 con output completo della Task 2 sono processati in questa fase
(mancano 2PCC e 1EWY). Vedi la sezione "Task 3" del report per la descrizione completa dell'algoritmo e per la
discussione dei risultati, incluso un limite metodologico nell'assegnazione delle proprietà biofisiche ai punti.

**Nota di correzione**: nella prima versione di `generate_interface_images.py`, il Canale 1 ("Shape Complementarity") usava il valore assoluto dello score, con convenzione di segno opposta al resto della pipeline (alto = peggiore, non migliore, complementarità). Corretto rimuovendo `.abs()`; i tensori sono stati rigenerati.

### 4.1 Planarità delle interfacce (proiezione PCA)

In [ ]:
pca_summary = pd.read_csv(REPO_DIR / "outputs" / "task3" / "summary_geometrico_pca_task3.csv")
print(f"Complessi processati: {len(pca_summary)}")
print(pca_summary['plane_variance_pct'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(pca_summary['plane_variance_pct'], bins=20, color="#4C72B0", alpha=0.8)
ax.axvline(pca_summary['plane_variance_pct'].mean(), color="black", linestyle="--",
           label=f"media = {pca_summary['plane_variance_pct'].mean():.1f}%")
ax.set_xlabel("Varianza spiegata dal piano medio, PC1+PC2 (%)")
ax.set_ylabel("Numero di complessi")
ax.set_title(f"Planarità delle interfacce ({len(pca_summary)} complessi)")
ax.legend()
plt.tight_layout()
plt.show()

### 4.2 Filtro energetico di Lennard-Jones: correlazione con l'affinità sperimentale

In [ ]:
from scipy.stats import pearsonr, spearmanr

lj_summary = pd.read_csv(REPO_DIR / "outputs" / "task3" / "summary_lennard_jones_affinity.csv")

r_geo, p_geo = pearsonr(lj_summary['pure_geometric_mean'], lj_summary['log_affinity'])
r_lj, p_lj = pearsonr(lj_summary['weighted_energy_geometric_mean'], lj_summary['log_affinity'])
rho_geo, ps_geo = spearmanr(lj_summary['pure_geometric_mean'], lj_summary['log_affinity'])
rho_lj, ps_lj = spearmanr(lj_summary['weighted_energy_geometric_mean'], lj_summary['log_affinity'])

print(f"n = {len(lj_summary)}")
print(f"Geometria pura:            Pearson R={r_geo:.4f} (p={p_geo:.4f})  Spearman rho={rho_geo:.4f} (p={ps_geo:.4f})")
print(f"Geometria + Lennard-Jones: Pearson R={r_lj:.4f} (p={p_lj:.4f})  Spearman rho={rho_lj:.4f} (p={ps_lj:.4f})")
print("\nNessuna delle due correlazioni è statisticamente significativa (p >> 0.05).")

### 4.3 Anteprima della mappa 2D multi-canale (complesso 1KTZ)

In [ ]:
tensor_1ktz = np.load(REPO_DIR / "outputs" / "task3" / "interface_maps" / "00000_1KTZ_tensor.npy")
print("Shape:", tensor_1ktz.shape)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
titles = ["Ch 1: Shape Complementarity", "Ch 2: Electrostatic Charge", "Ch 3: Hydrophobicity"]
cmaps = ["viridis", "bwr", "YlOrBr"]
vranges = [(tensor_1ktz[0].min(), tensor_1ktz[0].max()), (-1.0, 1.0), (-4.5, 4.5)]

for i in range(3):
    im = axes[i].imshow(tensor_1ktz[i], cmap=cmaps[i], origin="lower", vmin=vranges[i][0], vmax=vranges[i][1])
    axes[i].set_title(titles[i])
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
plt.suptitle("Mappa 2D multi-canale (32x32) — complesso 1KTZ", y=1.02)
plt.tight_layout()
plt.show()

## 5. Task 4 — Predizione dell'affinità tramite CNN

Script: `scripts/task4/models.py` (architettura `InterfaceCNN` + `ProteinInterfaceDataset`), `scripts/task4/train_final.py` (pipeline di training a due livelli, senza leakage: split di validazione interno per la selezione del checkpoint, standardizzazione Z-score calcolata solo sul training set di ciascun fold).

**Dataset**: dei 118 complessi disponibili dalla Task 3, 2 sono stati esclusi dopo un controllo di integrità (`1NVU`: tensore e target sperimentale relativi a catene ligando diverse; `1UUG`: affinità riportata come limite di rilevazione `<1E-13` M, non un valore numerico). Il training usa quindi **116 complessi**. Vedi la sezione "Task 4" del report per i dettagli completi.

### 5.1 Predizioni out-of-fold: metriche finali

In [ ]:
from scipy.stats import pearsonr, spearmanr

oof = pd.read_csv(REPO_DIR / "outputs" / "task4_final" / "oof_predictions.csv")
print(f"n = {len(oof)}")

rmse = ((oof['pkd_predicted'] - oof['pkd_real']) ** 2).mean() ** 0.5
r, p_r = pearsonr(oof['pkd_predicted'], oof['pkd_real'])
rho, p_rho = spearmanr(oof['pkd_predicted'], oof['pkd_real'])

print(f"RMSE:         {rmse:.4f}")
print(f"Pearson R:    {r:.4f}  (p={p_r:.4g})")
print(f"Spearman rho: {rho:.4f}  (p={p_rho:.4g})")
print(f"R^2:          {r**2:.4f} -> {r**2*100:.1f}% della varianza spiegata")

### 5.2 Predetto vs reale

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(oof['pkd_real'], oof['pkd_predicted'], alpha=0.6, color="#4C72B0", edgecolors='k', linewidths=0.3)

lims = [min(oof['pkd_real'].min(), oof['pkd_predicted'].min()) - 0.5,
        max(oof['pkd_real'].max(), oof['pkd_predicted'].max()) + 0.5]
ax.plot(lims, lims, 'r--', lw=1.5, label="Predizione perfetta")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel(r"$pK_D$ sperimentale")
ax.set_ylabel(r"$pK_D$ predetto (OOF)")
ax.set_title(f"CNN d'interfaccia: predizioni OOF vs affinità sperimentale (n={len(oof)})")
ax.legend()
ax.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

### 5.3 Architettura della rete

*(richiede `torch`; se non disponibile nell'ambiente, questa cella può essere saltata senza impatto sul resto del notebook — le metriche e il grafico sopra sono già calcolati dal solo `oof_predictions.csv`.)*

In [ ]:
import sys
sys.path.append(str(REPO_DIR / "scripts" / "task4"))

try:
    import torch
    from models import InterfaceCNN

    model = InterfaceCNN()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Numero totale di parametri: {n_params:,}")
    print(model)
except ImportError:
    print("torch non disponibile in questo ambiente: cella saltata.")

## 6. Discussione e conclusioni

*(da completare in linea con la Sezione "Discussione e limiti" del report)*